# "Irregular twice in a row" — the 30-second check on the Long-Term AF Database (Notebook 12)
**Bektur Asilov · COMSE-23 · Ala-Too**

CinC 2017 has one short recording per person, so it cannot tell what happens when a person repeats the check.
The Long-Term AF Database (PhysioNet, 84 people with paroxysmal or sustained AF, about 24 h each, 128 Hz, rhythm
labelled by experts) can: two consecutive 30-s stretches of one recording behave like a check and its repeat.
It was never used for development, so this is also a third external test of the v3 check.

Rules fixed **before** the run:
* lead 1 only, band-pass 0.5–40 Hz, resampled to 125 Hz; the server `check()` (v3) unchanged;
* every 2 minutes a **pair** of consecutive 30-s windows (t…t+30 s and t+30…t+60 s);
* a pair counts as **AF** if the whole minute is labelled `(AFIB`, as **normal** if the whole minute is `(N`;
  other rhythms and mixed minutes are left out;
* **single** rule = the verdict of the first window; **twice** rule = "irregular" only if both windows say
  irregular; "regular" if neither says irregular and at least one says regular; otherwise (one irregular and one
  regular, or both unreadable) "repeat";
  primary numbers: AF minutes called irregular, normal minutes called irregular, and minutes needing a repeat,
  for both rules; also per person (median over people) because minutes of one person are not independent.

> Kaggle: Internet **On** (about 1.5 GB from the PhysioNet open-data mirror), CPU.

In [ ]:
import os, time, json, subprocess, sys
import numpy as np, pandas as pd
from scipy.signal import butter, sosfilt, resample_poly
from multiprocessing import Pool
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
FS = 125
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
DB = os.environ.get('LTAF_DIR')
BASES = ['https://physionet-open.s3.amazonaws.com/ltafdb/1.0.0/', 'https://physionet.org/files/ltafdb/1.0.0/']
def get(f, dst):
    for b in BASES:
        if subprocess.run(['curl', '-fsSL', '--max-time', '1800', '-o', dst, b + f]).returncode == 0: return
    raise RuntimeError('download failed: ' + f)
if not DB:
    DB = '/kaggle/tmp/ltafdb'; os.makedirs(DB, exist_ok=True)
    get('RECORDS', f'{DB}/RECORDS')
RECS = [l.strip() for l in open(f'{DB}/RECORDS') if l.strip()]
def fetch_rec(r):
    for ext in ('hea', 'dat', 'atr'):
        if not os.path.exists(f'{DB}/{r}.{ext}'): get(f'{r}.{ext}', f'{DB}/{r}.{ext}')
    return r
T0 = time.time()
with Pool(8) as pool: pool.map(fetch_rec, RECS)
print(len(RECS), 'records downloaded in', f'{time.time()-T0:.0f} s')

## Server code, unchanged

In [ ]:
"""ECG strip analysis on the server: R-peaks, RR intervals, Poincaré points, signal quality index, PNG image.

Pure Python (strips are short: 10 s x 125 Hz). The R-peak detector is the firmware algorithm
(derivative^2 -> 150 ms moving average -> adaptive threshold, 250 ms refractory), adapted to run offline
on a short strip: the threshold tracker is primed with the first 2 s instead of a 2-s warm-up.
"""

import io
import math
import statistics


def _highpass(x: list[float], fs: int) -> list[float]:
    """Remove baseline wander: subtract a 0.6-s centred moving mean."""
    w = max(3, int(0.6 * fs)) | 1
    h = w // 2
    n = len(x)
    pre = [0.0]
    for v in x:
        pre.append(pre[-1] + v)
    out = []
    for i in range(n):
        a, b = max(0, i - h), min(n, i + h + 1)
        out.append(x[i] - (pre[b] - pre[a]) / (b - a))
    return out


def r_peaks(x: list[float], fs: int = 125) -> list[int]:
    n = len(x)
    if n < fs:
        return []
    y = _highpass(x, fs)
    ma = max(3, int(0.15 * fs))
    env, buf, s, prev = [], [0.0] * ma, 0.0, y[0]
    for i, v in enumerate(y):
        e = (v - prev) ** 2
        prev = v
        s += e - buf[i % ma]
        buf[i % ma] = e
        env.append(s / ma)
    decay = 0.5 ** (1 / (2.0 * fs))
    thr, refr = 0.35, int(0.25 * fs)
    pk = max(env[: min(n, 2 * fs)]) or 1e-12
    out, last, armed = [], -10 ** 9, True
    for i in range(int(0.2 * fs), n):
        pk = max(pk * decay, env[i])
        if armed and env[i] > thr * pk and i - last > refr:
            # the envelope lags the QRS by ~MA/2: take the largest |amplitude| just before
            a, b = max(0, i - ma - 4), min(n, i + 3)
            j = max(range(a, b), key=lambda k: abs(y[k]))
            if not out or j - out[-1] > refr:
                out.append(j)
            last, armed = i, False
        elif not armed and env[i] < 0.5 * thr * pk:
            armed = True
    return out


def rr_intervals(peaks: list[int], fs: int = 125) -> list[float]:
    return [round((b - a) / fs, 3) for a, b in zip(peaks, peaks[1:])]


def poincare(rr: list[float]) -> dict:
    """Points (RRn, RRn+1) and the SD1/SD2 descriptors (SD1 = short-term, beat-to-beat variability)."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    return poincare_points([[a, b] for a, b in zip(rr, rr[1:])])


def poincare_points(pts: list[list[float]]) -> dict:
    if len(pts) < 3:
        return {"points": pts, "sd1": None, "sd2": None, "ratio": None}
    d1 = [(b - a) / math.sqrt(2) for a, b in pts]
    d2 = [(b + a) / math.sqrt(2) for a, b in pts]
    sd1, sd2 = statistics.pstdev(d1), statistics.pstdev(d2)
    return {"points": pts, "sd1": round(sd1 * 1000, 1), "sd2": round(sd2 * 1000, 1),
            "ratio": round(sd1 / sd2, 2) if sd2 > 0 else None}


def irregularity(rr: list[float]) -> float | None:
    """Mean |successive difference| / mean RR — high (>~0.1) in AF, low in sinus rhythm."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    if len(rr) < 4:
        return None
    return round(statistics.mean(abs(b - a) for a, b in zip(rr, rr[1:])) / statistics.mean(rr), 3)


def quality(x: list[float], fs: int = 125, peaks: list[int] | None = None) -> dict:
    """Signal quality index 0..1 from four checks a cardiologist does by eye:
    no flat line / clipping, plausible heart rate, consistent QRS shape, little high-frequency noise."""
    n = len(x)
    if n < 2 * fs:
        return {"sqi": 0.0, "level": "poor", "reasons": ["too_short"]}
    peaks = r_peaks(x, fs) if peaks is None else peaks
    y = _highpass(x, fs)
    reasons = []
    rng = max(y) - min(y)
    flat = sum(1 for a, b in zip(y, y[1:]) if abs(b - a) < 1e-4) / n
    s_flat = 0.0 if rng < 0.05 or flat > 0.5 else 1.0
    if s_flat == 0:
        reasons.append("flat")
    secs = n / fs
    bpm = len(peaks) / secs * 60
    s_rate = 1.0 if 30 <= bpm <= 200 else 0.0
    if not s_rate:
        reasons.append("rate")
    # QRS template correlation
    h = int(0.1 * fs)
    segs = [y[p - h:p + h] for p in peaks if p - h >= 0 and p + h <= n]
    if len(segs) >= 3:
        tmpl = [statistics.median(col) for col in zip(*segs)]

        def corr(a, b):
            ma, mb = statistics.mean(a), statistics.mean(b)
            num = sum((u - ma) * (v - mb) for u, v in zip(a, b))
            den = math.sqrt(sum((u - ma) ** 2 for u in a) * sum((v - mb) ** 2 for v in b)) or 1e-12
            return num / den
        s_shape = max(0.0, statistics.mean(corr(s, tmpl) for s in segs))
    else:
        s_shape = 0.0
    if s_shape < 0.7:
        reasons.append("shape")
    # high-frequency noise: energy of the second difference relative to the signal
    d2 = [y[i + 1] - 2 * y[i] + y[i - 1] for i in range(1, n - 1)]
    noise = (sum(v * v for v in d2) / len(d2)) ** 0.5 / (statistics.pstdev(y) or 1e-12)
    s_noise = max(0.0, min(1.0, 1.5 - noise))
    if s_noise < 0.5:
        reasons.append("noise")
    sqi = s_flat * s_rate * (0.6 * s_shape + 0.4 * s_noise)
    sqi = round(max(0.0, min(1.0, sqi)), 2)
    return {"sqi": sqi, "level": "good" if sqi >= 0.7 else "fair" if sqi >= 0.4 else "poor", "reasons": reasons}


PAUSE_S = 3.0   # an RR gap of 3 s or more is a pause worth showing to a doctor


def pauses(peaks: list[int], fs: int = 125) -> list[dict]:
    return [{"at_s": round(a / fs, 2), "rr_s": round((b - a) / fs, 2)}
            for a, b in zip(peaks, peaks[1:]) if (b - a) / fs >= PAUSE_S]


def analyze(x: list[float], fs: int = 125) -> dict:
    p = r_peaks(x, fs)
    rr = rr_intervals(p, fs)
    q = quality(x, fs, p)
    # a pause only counts when the rest of the strip is clean; otherwise it is more likely a lost electrode
    return {"r_peaks": p, "rr": rr, "poincare": poincare(rr), "irregularity": irregularity(rr),
            "quality": q, "pauses": pauses(p, fs) if q["sqi"] >= 0.4 else []}


def _movmean(x: list[float], w: int) -> list[float]:
    """Centred moving mean over w samples (w odd), shrinking at the edges — the same as in _highpass."""
    h, n = w // 2, len(x)
    pre = [0.0]
    for v in x:
        pre.append(pre[-1] + v)
    return [(pre[min(n, i + h + 1)] - pre[max(0, i - h)]) / (min(n, i + h + 1) - max(0, i - h)) for i in range(n)]


# second detector settings (smooth samples, baseline window s, threshold fraction, refractory s)
AMP_DEFAULT = (3, 0.12, 0.5, 0.30)   # v3: best F1 on MIT-BIH of 16 settings (Notebook 9); v2 was (1, 0.6, 0.5, 0.25)


def r_peaks_amp(x: list[float], fs: int = 125, params: tuple = AMP_DEFAULT) -> list[int]:
    """Second, independent R-peak detector for the agreement check: amplitude of the signal after removing
    the baseline (not the slope energy the main detector uses), adaptive threshold at a fraction of the
    decaying peak, refractory period. Optional light smoothing and a short baseline window make it a crude
    QRS band-pass that ignores T-waves better."""
    w1, w2s, frac, refr_s = params
    n = len(x)
    if n < fs:
        return []
    y = _movmean(x, w1) if w1 > 1 else list(x)
    w2 = max(3, int(w2s * fs)) | 1
    base = _movmean(y, w2)
    a = [abs(u - b) for u, b in zip(y, base)]
    decay = 0.5 ** (1 / (2.0 * fs))
    refr, look = int(refr_s * fs), int(0.1 * fs)
    pk = max(a[: min(n, 2 * fs)]) or 1e-12
    out, i = [], 0
    while i < n:
        pk = max(pk * decay, a[i])
        if a[i] > frac * pk and (not out or i - out[-1] > refr):
            j = max(range(i, min(n, i + look)), key=lambda k: a[k])
            if not out or j - out[-1] > refr:
                out.append(j)
            i = j + 1
            continue
        i += 1
    return out


def bsqi(p1: list[int], p2: list[int], fs: int = 125, tol_s: float = 0.15) -> float:
    """Agreement of two detectors (Li, Mark & Clifford 2008): matched / (n1 + n2 - matched), match within 150 ms."""
    if not p1 and not p2:
        return 0.0
    tol, used, m = int(tol_s * fs), set(), 0
    j = 0
    for p in p1:
        while j < len(p2) and p2[j] < p - tol:
            j += 1
        k = j
        while k < len(p2) and p2[k] <= p + tol:
            if k not in used:
                used.add(k)
                m += 1
                break
            k += 1
    return round(m / (len(p1) + len(p2) - m), 3)


# "Check me in 30 seconds" (wellness wording), see ml/CHECK30_CINC2017.md and ml/CHECK30_BSQI.md
# v1 (CinC 2017, never tuned on it): 91.4 % of AF recordings irregular, 14.1 % of normal ones too; noise not caught.
# v2 adds the two-detector agreement gate (bSQI >= 0.8), rules fixed before Notebook 8: false "irregular" on clean
# normal rhythm 14.1 % -> 3.3 %, noise caught (97.5 % of noisy recordings -> "could not read"), but 44 % of clean
# normal checks need a repeat and AF found drops 91.4 -> 69.4 %. See ml/CHECK30_BSQI.md.
# v3 (current): second detector chosen on MIT-BIH (Notebook 9). CinC 2017 clean: AF found 85.6 %, normal called
# irregular 8.9 %, repeats 11 %; catches noise less well than v2. See ml/CHECK30_V3.md.
CHECK_MIN_BEATS, CHECK_MIN_SQI, CHECK_IRR, CHECK_MIN_BSQI = 16, 0.4, 0.10, 0.8


def check(x: list[float], fs: int = 125, gate: bool = True) -> dict:
    """Verdict for a 30-s spot check: "regular", "irregular" (show a doctor) or "no_verdict" (repeat)."""
    a = analyze(x, fs)
    n, sqi, irr = len(a["r_peaks"]), a["quality"]["sqi"], a["irregularity"]
    b = bsqi(a["r_peaks"], r_peaks_amp(x, fs), fs)
    if n < CHECK_MIN_BEATS or sqi < CHECK_MIN_SQI or irr is None or (gate and b < CHECK_MIN_BSQI):
        verdict = "no_verdict"
    else:
        verdict = "irregular" if irr > CHECK_IRR else "regular"
    bpm = round(len(a["r_peaks"]) / (len(x) / fs) * 60) if x else None
    return {"verdict": verdict, "beats": n, "sqi": sqi, "irregularity": irr, "bsqi": b, "bpm": bpm,
            "reasons": a["quality"]["reasons"] + (["detectors_disagree"] if gate and b < CHECK_MIN_BSQI else [])}

In [ ]:
print('check v3: second detector', AMP_DEFAULT, '| gate bSQI >=', CHECK_MIN_BSQI)
def to_125(sig, fs):
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

def rhythm_at(ann, n):
    # rhythm label for every sample index (original fs); '' before the first rhythm annotation
    lab = np.empty(n, dtype=object); lab[:] = ''
    starts = [(s, a.strip().rstrip('\x00')) for s, a in zip(ann.sample, ann.aux_note) if a and a.startswith('(')]
    for k, (s, a) in enumerate(starts):
        e = starts[k + 1][0] if k + 1 < len(starts) else n
        lab[s:e] = a
    return lab

def job(rec):
    r = wfdb.rdrecord(f'{DB}/{rec}', channels=[0]); ann = wfdb.rdann(f'{DB}/{rec}', 'atr')
    fs = r.fs; sig = r.p_signal[:, 0]; n = len(sig)
    lab = rhythm_at(ann, n)
    out = []
    step, w = int(120 * fs), int(30 * fs)
    for t in range(0, n - 2 * w, step):
        seg = lab[t:t + 2 * w]
        if (seg == '(AFIB').all(): cls = 'AF'
        elif (seg == '(N').all(): cls = 'N'
        else: continue
        v = []
        for k in (0, 1):
            x = list(map(float, to_125(sig[t + k * w:t + (k + 1) * w], fs)))
            v.append(check(x, FS)['verdict'])
        out.append(dict(record=rec, t_s=t / fs, cls=cls, v1=v[0], v2=v[1]))
    return out

T0 = time.time(); rows = []
with Pool(os.cpu_count()) as pool:
    for i, part in enumerate(pool.imap_unordered(job, RECS)):
        rows += part
        if (i + 1) % 10 == 0: print(f'{i+1}/{len(RECS)} records [{time.time()-T0:.0f} s]', flush=True)
D = pd.DataFrame(rows); D.to_csv(f'{OUT_DIR}/ltafdb_pairs.csv', index=False)
print(len(D), 'minute pairs;', D.cls.value_counts().to_dict(), 'people:', D.record.nunique())

## Results

In [ ]:
def twice(a, b):
    if a == 'irregular' and b == 'irregular': return 'irregular'
    if 'regular' in (a, b) and 'irregular' not in (a, b): return 'regular'
    return 'no_verdict'          # disagreement (one irregular, one regular) or both unreadable -> repeat again
D['single'] = D.v1
D['twice'] = [twice(a, b) for a, b in zip(D.v1, D.v2)]
def shares(g, col):
    v = g[col].value_counts(normalize=True)
    return {k: round(float(v.get(k, 0)), 4) for k in ('irregular', 'no_verdict', 'regular')}
S = {'pairs': len(D), 'people': int(D.record.nunique()),
     'people_with_AF_minutes': int(D[D.cls == 'AF'].record.nunique()), 'people_with_N_minutes': int(D[D.cls == 'N'].record.nunique())}
for cls in ('AF', 'N'):
    g = D[D.cls == cls]
    S[cls] = {'minutes': len(g), 'single': shares(g, 'single'), 'twice': shares(g, 'twice')}
    per = g.groupby('record').apply(lambda h: pd.Series({'single_irr': (h.single == 'irregular').mean(), 'twice_irr': (h.twice == 'irregular').mean()}))
    S[cls]['per_person_median'] = per.median().round(4).to_dict()
json.dump(S, open(f'{OUT_DIR}/ltafdb_summary.json', 'w'), indent=1, default=float)
print(json.dumps(S, indent=1))